# Gold: dim_products

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql
from src.gold.dims.products import build_dim_products_scd2, build_incremental_dim_products, validate_scd2_products
from src.watermark import  get_last_commit_ts, get_effective_watermark, get_changed_keys
from src.writers import overwrite_table, replace_by_key

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
GOLD_NAMESPACE = cfg["general"]["namespaces"]["gold"]

cfg_product= cfg["gold"]["dim_products_scd2"]
SOURCE_TABLE = cfg_product["source_table"]
TARGET_TABLE = cfg_product["target_table"]
TRANSLATION_TABLE = cfg_product["translation_table"]
BUFFER_HOURS = cfg_product["buffer_hours"]
KEY_COLUMN = cfg_product["key_column"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["gold_products"])
        .getOrCreate()
)

## Run Pipeline

In [4]:
def run_dim_products_pipeline(spark):
    print("[START] dim_products pipeline")

    last_commit_ts = get_last_commit_ts(spark, TARGET_TABLE)
    print(f"[INFO] last_commit_ts = {last_commit_ts}")

    if last_commit_ts is None:
        print("[INFO] first run → full rebuild")
        df = build_dim_products_scd2(spark, SOURCE_TABLE, TRANSLATION_TABLE)
        validate_scd2_products(df)
        overwrite_table(df, TARGET_TABLE)
        print("[END] full rebuild complete")
        return

    effective_ts = get_effective_watermark(last_commit_ts, BUFFER_HOURS)
    changed_product_ids = get_changed_keys(spark, SOURCE_TABLE, KEY_COLUMN, effective_ts)

    if changed_product_ids.isEmpty():
        print("[INFO] no changes detected → skip")
        return

    print("[INFO] changes detected → incremental run")
    df = build_incremental_dim_products(spark, SOURCE_TABLE, TRANSLATION_TABLE, changed_product_ids)
    validate_scd2_products(df)
    replace_by_key(spark, df, TARGET_TABLE, KEY_COLUMN)
    print("[END] incremental update complete")

In [5]:
# if __name__ == "__main__":
#     from pyspark.sql import SparkSession

#     spark = SparkSession.builder.getOrCreate()
run_dim_products_pipeline(spark)

[START] dim_products pipeline
[INFO] last_commit_ts = None
[INFO] first run → full rebuild


[END] full rebuild complete


## Sanity Check

In [6]:
%%sql
SHOW TABLES IN polaris.gold;

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|gold     |dim_customers_scd2|false      |
|gold     |dim_sellers_scd2  |false      |
|gold     |dim_products_scd2 |false      |
+---------+------------------+-----------+



In [7]:
%%sql
SELECT * FROM polaris.gold.dim_products_scd2
LIMIT 10

+--------------------------------+---------------------+-----------------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+------------------+-----------------------+-----------------------+-----------------------+------------+----------+----------------------------------------------------------------+
|product_id                      |product_category_name|product_category_name_english|product_name_length|product_description_length|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_volume_cm3|cdc_ts                 |spark_ingest_ts        |effective_from         |effective_to|is_current|product_sk                                                      |
+--------------------------------+---------------------+-----------------------------+-------------------+--------------------------+------------------+----------------+-----------------+-------

In [8]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop() 